# Gold analytics outputs

Build resource counts and practical resource-specific summary tables from the validated Silver snapshots.

In [ ]:
from pyspark.sql import functions as F
resources = ["patient", "encounter", "observation", "condition"]
available = [r for r in resources if spark.catalog.tableExists(f"silver_{r}")]
if not available:
    raise ValueError("No Silver tables found. Run Silver transformations first.")

summary_frames = []
for r in available:
    summary_frames.append(spark.table(f"silver_{r}").agg(
        F.lit(r.title()).alias("resource_type"),
        F.count("*").alias("record_count"),
        F.countDistinct("resource_id").alias("distinct_resource_count")
    ))
summary = summary_frames[0]
for frame in summary_frames[1:]:
    summary = summary.unionByName(frame)
summary.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold_resource_summary")

if spark.catalog.tableExists("silver_observation"):
    obs = spark.table("silver_observation")
    (obs.groupBy("code","code_display","status")
        .agg(F.count("*").alias("observation_count"),
             F.avg("value_quantity").alias("average_numeric_value"),
             F.count("value_quantity").alias("numeric_value_count"))
        .write.format("delta").mode("overwrite").option("overwriteSchema","true")
        .saveAsTable("gold_observation_summary"))

if spark.catalog.tableExists("silver_encounter"):
    (spark.table("silver_encounter").groupBy("status","class_code")
        .agg(F.count("*").alias("encounter_count"))
        .write.format("delta").mode("overwrite").option("overwriteSchema","true")
        .saveAsTable("gold_encounter_summary"))

if spark.catalog.tableExists("silver_condition"):
    (spark.table("silver_condition").groupBy("code","code_display","clinical_status")
        .agg(F.count("*").alias("condition_count"))
        .write.format("delta").mode("overwrite").option("overwriteSchema","true")
        .saveAsTable("gold_condition_summary"))

print("Gold summary tables created for available Silver resources:", available)
